# Final Project Report
## Portfolio Risk Management System

---

**Course:** MATH5320

**Project:** Financial Risk Management System

**Repository:** https://github.com/nl2992/MATH5320

**Team Members:** Nigel Li, Michael Adegbite, Stella

---

### Table of Contents
1. Introduction
2. Portfolio and Data
3. Risk Models
   - 3.1 Historical VaR and ES
   - 3.2 Parametric VaR and ES
   - 3.3 Monte Carlo VaR and ES
4. Option Pricing
5. Backtesting
6. Software Design
7. Test Plan
8. Test Results and Analysis
9. Limitations
10. Conclusion

---

### 1. Introduction

This project develops a portfolio risk management system designed to measure and analyze potential losses in financial portfolios consisting of equities and derivative instruments. In modern financial markets, accurate risk estimation is essential for decision-making, regulatory compliance, and capital allocation. As such, this system focuses on implementing and comparing widely used risk metrics, including Value at Risk (VaR) and Expected Shortfall (ES), across multiple methodologies.

The system integrates three primary approaches to risk measurement: historical simulation, parametric (variance–covariance), and Monte Carlo simulation. Each method provides a different perspective on risk, allowing for a more comprehensive assessment of portfolio exposure under varying assumptions about market behavior. In addition, the inclusion of European option pricing using the Black–Scholes framework enables the system to handle portfolios with nonlinear payoffs.

Beyond risk computation, the project emphasizes validation and reliability through backtesting procedures. By comparing model-generated risk estimates with realized portfolio outcomes, the system evaluates the accuracy and robustness of each method. The implementation is designed as a modular and extensible framework, supported by a user-friendly interface, enabling efficient analysis and practical application in a real-world risk management context.

---

### 2. Portfolio and Data

This system is designed to analyze portfolios consisting of equities and European options. The portfolio framework allows users to input multiple assets, including individual stocks and option positions, enabling the evaluation of both linear and nonlinear risk exposures. For option positions, values are computed using the Black–Scholes pricing model, ensuring consistency in valuation across the portfolio.

The primary data used in this project consists of historical price data for the underlying assets. From these prices, log-returns are computed and used as the basis for all risk calculations. The system supports flexible time horizons, allowing users to select different lookback windows depending on the analysis. This enables the comparison of short-term and long-term risk estimates.

For parametric methods, statistical parameters such as mean returns and covariance matrices are estimated directly from the historical data. For Monte Carlo simulation, the same estimated parameters are used to generate simulated return paths under the assumption of normally distributed returns. All data preprocessing steps, including cleaning, return computation, and parameter estimation, are handled within the system to ensure consistency across all risk models.

Overall, the data pipeline is designed to be simple, flexible, and fully integrated with the risk models, allowing for efficient and reproducible portfolio risk analysis.

---

### 3. Risk Models

This project implements multiple approaches to measure portfolio risk, focusing on Value at Risk (VaR) and Expected Shortfall (ES).

Value at Risk (VaR) at confidence level α is defined as:

VaR_α = − Quantile_{1−α}(R)

Expected Shortfall (ES) is defined as:

ES_α = − E[ R | R ≤ −VaR_α ]

To capture different perspectives on risk, three methodologies are implemented: historical simulation, parametric estimation, and Monte Carlo simulation.

---

#### 3.1 Historical VaR and Expected Shortfall

The historical simulation approach estimates risk directly from past observed returns without assuming any specific distribution.

VaR is computed as the empirical quantile of historical returns:

VaR_α = − Quantile_{1−α}(R)

Expected Shortfall is calculated as the average of losses beyond the VaR threshold:

ES_α = − average of returns such that R ≤ −VaR_α

This method captures real market behavior, including skewness and fat tails, but assumes that historical patterns will persist in the future.

---

#### 3.2 Parametric VaR and Expected Shortfall

The parametric approach assumes that portfolio returns follow a normal distribution:

R ~ N(μ, σ²)

VaR is computed as:

VaR_α = − ( μ + z_α σ )

where z_α is the standard normal quantile.

Expected Shortfall is given by:

ES_α = − ( μ + σ * φ(z_α) / (1 − α) )

where φ(·) is the standard normal density function.

This method is computationally efficient but may underestimate risk when returns exhibit non-normal features such as heavy tails or skewness.

---

#### 3.3 Monte Carlo VaR and Expected Shortfall

The Monte Carlo simulation approach generates a large number of simulated return scenarios based on estimated parameters.

Simulated returns are generated as:

R_sim = μ + σZ, where Z ~ N(0,1)

A large number of simulations are used to construct the distribution of portfolio returns.

VaR and Expected Shortfall are then computed from the simulated distribution:

VaR_α = − Quantile_{1−α}(R_sim)

ES_α = − E[ R_sim | R_sim ≤ −VaR_α ]

This method is highly flexible and can handle nonlinear payoffs, making it suitable for portfolios containing options, but it is computationally more intensive.

---

### 4. Option Pricing

To accurately evaluate portfolios containing derivative instruments, this project incorporates option pricing using the Black–Scholes model. This framework provides a consistent method for valuing European call and put options based on underlying asset prices and market parameters.

The price of a European call option is given by:

C = S₀ N(d₁) − K e^(−rT) N(d₂)

The price of a European put option is given by:

P = K e^(−rT) N(−d₂) − S₀ N(−d₁)

where:

d₁ = [ ln(S₀ / K) + (r + 0.5 σ²)T ] / (σ√T)
d₂ = d₁ − σ√T

S₀ = current stock price
K = strike price
r = risk-free interest rate
σ = volatility
T = time to maturity
N(·) = cumulative distribution function of the standard normal distribution

This model is used to compute option values within the portfolio, allowing the system to capture nonlinear payoffs when calculating risk measures such as VaR and Expected Shortfall. While the Black–Scholes model assumes constant volatility and lognormally distributed prices, it provides a practical and widely used benchmark for option valuation in risk management applications.

---

### 5. Backtesting

To evaluate the accuracy and reliability of the risk models, backtesting is performed by comparing predicted Value at Risk (VaR) with realized portfolio returns. Backtesting assesses whether the frequency of observed losses exceeding VaR aligns with the chosen confidence level.

For a given confidence level α, the expected proportion of VaR violations (exceptions) is (1 − α). Let N be the total number of observations and x be the number of observed exceptions. A model is considered accurate if x is consistent with the expected number of violations.

To formally test this, the Kupiec Proportion of Failures (POF) test is applied. The test statistic is given by:

LR = −2 ln [ ((1 − α)^(N − x) * α^x) / ((1 − x/N)^(N − x) * (x/N)^x) ]

This statistic follows a chi-square distribution with one degree of freedom. If the test statistic exceeds the critical value, the model is rejected, indicating that the VaR estimates are not consistent with observed outcomes.

Backtesting is performed across different risk models to compare their performance. A model that produces too many exceptions underestimates risk, while one with too few exceptions may be overly conservative. This process provides a quantitative way to validate the effectiveness of each risk estimation method.

---

### 6. Software Design

The system is designed as a modular and extensible framework for portfolio risk analysis. The overall architecture separates data processing, risk modeling, and user interaction, ensuring clarity, reusability, and ease of maintenance.

The workflow of the system follows three main stages:

1. Data Input and Preprocessing
   Historical price data is loaded and cleaned, and returns are computed. For portfolios containing options, relevant parameters such as volatility, interest rates, and time to maturity are incorporated.

2. Risk Model Implementation
   The system implements multiple risk models, including historical, parametric, and Monte Carlo approaches. Each model is structured as an independent component, allowing for consistent input handling and direct comparison of results across methods.

3. Output and Visualization
   Risk metrics such as VaR and Expected Shortfall are computed and presented to the user. The system includes a Streamlit-based interface that allows users to interactively select portfolios, adjust parameters, and visualize results.

The codebase is organized to promote modularity, with separate components for data handling, model computation, and application logic. This structure allows for easy extension of the system, such as adding new risk models or incorporating more advanced assumptions. Additionally, the design ensures consistency across all calculations, as all models rely on a shared data pipeline and parameter estimation process.

---

### 7. Test Plan

A structured testing approach is used to ensure the correctness, consistency, and reliability of the risk management system. The testing process focuses on validating both individual components and the system as a whole.

First, unit tests are applied to core functions, including return calculations, parameter estimation, and risk metric computations (VaR and Expected Shortfall). These tests verify that each component produces correct outputs for known inputs.

Second, integration testing is performed to ensure that different parts of the system work together correctly. This includes verifying that data flows properly from preprocessing through to risk model calculations and final outputs.

Third, consistency checks are conducted across different risk models. For example, VaR and Expected Shortfall values are compared across historical, parametric, and Monte Carlo methods to ensure results are reasonable and aligned with theoretical expectations.

Finally, backtesting is used as a validation tool to assess model performance over time. By comparing predicted VaR with actual realized losses, the system evaluates whether each model produces reliable risk estimates.

This multi-level testing approach ensures that the system is robust, accurate, and suitable for practical risk analysis applications.

---

### 8. Test Results and Analysis

The performance of the risk models is evaluated by comparing Value at Risk (VaR) and Expected Shortfall (ES) estimates across the historical, parametric, and Monte Carlo approaches. Each method provides a different perspective on portfolio risk, reflecting its underlying assumptions.

The historical method captures empirical market behavior and typically produces higher risk estimates during periods of market stress due to its ability to incorporate extreme past observations. The parametric approach, which assumes normally distributed returns, tends to produce smoother and often lower risk estimates, as it does not fully account for heavy tails or skewness in the data. The Monte Carlo method provides a flexible alternative, generating simulated scenarios that allow for a more comprehensive view of potential outcomes, particularly for portfolios with nonlinear exposures.

Backtesting results are used to assess model accuracy. The number of VaR exceptions is compared to the expected number based on the chosen confidence level. Models that produce too many exceptions are considered to underestimate risk, while those with too few may be overly conservative. The Kupiec test provides a formal statistical framework for evaluating whether the observed exception rate is consistent with model assumptions.

Overall, the results highlight the trade-offs between simplicity, accuracy, and computational efficiency. The parametric method is efficient but relies on strong assumptions, the historical method reflects real data but depends on past observations, and the Monte Carlo method offers flexibility at the cost of increased computational effort. These findings demonstrate the importance of using multiple approaches when assessing portfolio risk.

---

### 9. Limitations

While the system provides a comprehensive framework for portfolio risk analysis, several limitations should be noted.

First, the parametric approach assumes that returns are normally distributed. In reality, financial returns often exhibit skewness and heavy tails, which can lead to an underestimation of extreme risk.

Second, the historical simulation method relies entirely on past data. This assumes that historical market conditions will persist in the future, which may not hold during periods of structural change or unprecedented market events.

Third, the Monte Carlo simulation is based on simplified assumptions about return dynamics, typically using constant mean and volatility. This does not capture more complex features of financial markets such as volatility clustering or time-varying risk.

Fourth, the Black–Scholes model used for option pricing assumes constant volatility and frictionless markets. These assumptions may not reflect real-world conditions, particularly during periods of market stress.

Finally, the system does not explicitly incorporate advanced risk factors such as stochastic volatility, liquidity risk, or correlations that change over time. As a result, the risk estimates should be interpreted as approximations rather than exact measures of future losses.

Despite these limitations, the system provides a solid foundation for understanding and comparing different approaches to portfolio risk management.

---

### 10. Conclusion

This project develops a portfolio risk management system that implements and compares multiple approaches to measuring financial risk. By integrating historical simulation, parametric methods, and Monte Carlo simulation, the system provides a comprehensive framework for evaluating Value at Risk (VaR) and Expected Shortfall (ES) under different assumptions.

In addition to risk computation, the project incorporates option pricing through the Black–Scholes model, allowing for the analysis of portfolios with nonlinear payoffs. The inclusion of backtesting and the Kupiec test ensures that the models are not only implemented correctly but also validated against observed market behavior.

The results highlight the strengths and limitations of each approach. While simpler models offer computational efficiency, more flexible methods provide a deeper understanding of tail risk. This reinforces the importance of using multiple models in practice rather than relying on a single approach.

Overall, the system demonstrates how quantitative risk models can be implemented in a structured and practical way. It provides a foundation for further extensions, such as incorporating more advanced models, improving assumptions, or expanding to additional asset classes.

---